# Electricity Consumption Analysis and Demand Prediction Using Data Analytics and Artificial Intelligence

**Student:** Swetha V  
**Degree:** B.E. Computer Science and Engineering  
**Program:** AICTE × IBM SkillsBuild Data Analytics with AI Internship  
**Organization:** BharatCares

## Project Overview
This project analyzes historical household electricity measurements and builds machine-learning regression models to predict hourly electricity demand.

The project uses the **Individual Household Electric Power Consumption** dataset from the UCI Machine Learning Repository (Dataset 235). The original data contains one-minute measurements over almost four years.

> **Important:** This notebook is designed to use the real UCI dataset. It does not contain invented results. Numerical findings, model metrics, and plots appear only after the notebook is executed with the dataset.

## 1. Problem Statement

Electricity consumption changes over time because household activities are not the same throughout the day, week, or year. Without systematic analysis, it is difficult to identify high-demand periods, understand usage patterns, and estimate future electricity demand.

This project uses data analytics and machine learning to clean historical electricity data, discover consumption patterns, engineer time-based and lag-based features, and predict hourly household electricity demand.

## 2. Objectives

1. Analyze historical electricity consumption data.
2. Identify and handle missing or invalid observations.
3. Create useful date/time features.
4. Explore hourly, monthly, and weekday/weekend consumption patterns.
5. Study relationships among electrical measurements.
6. Build Linear Regression and Random Forest Regression models.
7. Evaluate models using MAE, RMSE, and R².
8. Compare actual and predicted demand.
9. Generate findings and practical recommendations from the actual results.

## 3. Dataset Information

**Dataset:** Individual Household Electric Power Consumption  
**Source:** UCI Machine Learning Repository, Dataset 235  
**Official source:** https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption

The main columns are:

| Column | Meaning |
|---|---|
| Date | Calendar date |
| Time | Time of measurement |
| Global_active_power | Household active power in kilowatts |
| Global_reactive_power | Reactive power in kilowatts |
| Voltage | Average voltage in volts |
| Global_intensity | Current intensity in amperes |
| Sub_metering_1 | Kitchen-related sub-metering |
| Sub_metering_2 | Laundry-room-related sub-metering |
| Sub_metering_3 | Water-heater/air-conditioner-related sub-metering |

### Prediction target
The project predicts **hourly mean Global_active_power**. In simple words, the target tells us the average household active power demand during an hour.

## 4. Project Workflow

Dataset → Data Collection → Data Understanding → Data Cleaning → Preprocessing → EDA → Visualization → Feature Engineering → Chronological Train/Test Split → Machine Learning → Evaluation → Prediction → Insights → Recommendations

## 5. Import Libraries

**Why?** These libraries provide tools for reading data, calculations, visualization, and machine learning.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
DATA_DIR.mkdir(exist_ok=True)

DATA_FILE = DATA_DIR / "household_power_consumption.txt"
ZIP_FILE = DATA_DIR / "individual_household_electric_power_consumption.zip"

print("Working folder:", BASE_DIR)

## 6. Download the Real UCI Dataset

This cell downloads the official dataset automatically if the text file is not already present.

If your internet connection is unavailable, download the dataset manually from the official UCI page and place `household_power_consumption.txt` inside the project's `dataset` folder.

In [ ]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

if not DATA_FILE.exists():
    print("Dataset not found. Downloading from UCI...")
    urllib.request.urlretrieve(UCI_ZIP_URL, ZIP_FILE)

    with zipfile.ZipFile(ZIP_FILE, "r") as z:
        members = z.namelist()
        target_member = next(
            (m for m in members if m.endswith("household_power_consumption.txt")),
            None
        )
        if target_member is None:
            raise FileNotFoundError("The expected UCI dataset file was not found in the ZIP archive.")
        with z.open(target_member) as src, open(DATA_FILE, "wb") as dst:
            dst.write(src.read())

    print("Dataset downloaded and extracted:", DATA_FILE)
else:
    print("Dataset already exists:", DATA_FILE)

## 7. Load the Dataset

The original file uses semicolons as separators and `?` for missing values.

We load the columns as strings first so that missing-value handling is explicit.

In [ ]:
raw = pd.read_csv(
    DATA_FILE,
    sep=";",
    na_values="?",
    low_memory=False
)

print("Rows:", raw.shape[0])
print("Columns:", raw.shape[1])
raw.head()

## 8. Data Understanding

Before changing anything, inspect the structure of the data. This prevents accidental assumptions.

In [ ]:
print("Shape:", raw.shape)
print("\nColumn names:")
print(raw.columns.tolist())

print("\nData types:")
print(raw.dtypes)

print("\nMissing values:")
print(raw.isna().sum())

print("\nDuplicate rows:", raw.duplicated().sum())

print("\nSummary statistics:")
display(raw.describe(include="all").T)

print("\nFirst five records:")
display(raw.head())

## 9. Data Cleaning

### What are we doing?

1. Combine `Date` and `Time` into one datetime value.
2. Convert measurement columns to numeric values.
3. Sort records chronologically.
4. Remove duplicate timestamps if any exist.
5. Handle missing numeric measurements by time interpolation, followed by forward/backward filling only for any remaining edge gaps.

We do not blindly delete missing rows because the UCI documentation states that the dataset contains missing measurement values.

In [ ]:
df = raw.copy()

measurement_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    dayfirst=True,
    errors="coerce"
)

for col in measurement_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Datetime"]).copy()
df = df.sort_values("Datetime").drop_duplicates(subset="Datetime").reset_index(drop=True)

df.set_index("Datetime", inplace=True)

missing_before = df[measurement_cols].isna().sum()

df[measurement_cols] = (
    df[measurement_cols]
    .interpolate(method="time", limit_direction="both")
)

missing_after = df[measurement_cols].isna().sum()

print("Missing values before interpolation:")
display(missing_before.to_frame("Missing_Before"))

print("Missing values after interpolation:")
display(missing_after.to_frame("Missing_After"))

print("Cleaned shape:", df.shape)

## 10. Data Preprocessing and Hourly Aggregation

The original data is measured every minute. For this student-friendly prediction task, we convert it to hourly observations.

For the electrical measurements, hourly mean values are used. This gives a manageable time series while preserving the main demand pattern.

In [ ]:
hourly = df[measurement_cols].resample("h").mean()

print("Hourly dataset shape:", hourly.shape)
display(hourly.head())

## 11. Exploratory Data Analysis

EDA means **Exploratory Data Analysis**. In simple words, it means looking at the data carefully to understand patterns before building the model.

### 11.1 Basic statistics

In [ ]:
print("Average hourly active power (kW):", hourly["Global_active_power"].mean())
print("Maximum hourly active power (kW):", hourly["Global_active_power"].max())
print("Minimum hourly active power (kW):", hourly["Global_active_power"].min())

display(hourly["Global_active_power"].describe().to_frame("Global_active_Power"))

### 11.2 Electricity consumption over time

This graph shows how household active power changes throughout the dataset period. Because there are many hourly points, the chart also includes a 7-day rolling mean to make the longer-term pattern easier to see.

In [ ]:
target = hourly["Global_active_power"]

plt.figure(figsize=(14, 5))
plt.plot(hourly.index, target, alpha=0.25, linewidth=0.5, label="Hourly mean")
plt.plot(target.rolling(24 * 7).mean(), linewidth=2, label="7-day rolling mean")
plt.title("Hourly Household Electricity Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")
plt.legend()
plt.tight_layout()
plt.show()

### 11.3 Average consumption by hour

This helps identify which hours of the day usually have higher or lower demand.

In [ ]:
hourly["Hour"] = hourly.index.hour
avg_by_hour = hourly.groupby("Hour")["Global_active_power"].mean()

display(avg_by_hour.to_frame("Average_Active_Power_kW"))

plt.figure(figsize=(10, 5))
avg_by_hour.plot(kind="bar")
plt.title("Average Electricity Demand by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average Global Active Power (kW)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 11.4 Monthly consumption pattern

In [ ]:
hourly["Month"] = hourly.index.month
monthly_avg = hourly.groupby("Month")["Global_active_power"].mean()

display(monthly_avg.to_frame("Average_Active_Power_kW"))

plt.figure(figsize=(10, 5))
monthly_avg.plot(marker="o")
plt.title("Average Electricity Demand by Month")
plt.xlabel("Month")
plt.ylabel("Average Global Active Power (kW)")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

### 11.5 Weekday versus weekend

In [ ]:
hourly["Day_of_Week"] = hourly.index.dayofweek
hourly["Day_Name"] = hourly.index.day_name()
hourly["Is_Weekend"] = (hourly["Day_of_Week"] >= 5).astype(int)

weekday_weekend = hourly.groupby("Is_Weekend")["Global_active_power"].mean()
weekday_weekend.index = ["Weekday", "Weekend"]

display(weekday_weekend.to_frame("Average_Active_Power_kW"))

plt.figure(figsize=(7, 5))
weekday_weekend.plot(kind="bar")
plt.title("Weekday vs Weekend Electricity Demand")
plt.xlabel("Day Type")
plt.ylabel("Average Global Active Power (kW)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 11.6 Distribution of electricity consumption

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(hourly["Global_active_power"], bins=50, kde=True)
plt.title("Distribution of Hourly Global Active Power")
plt.xlabel("Global Active Power (kW)")
plt.ylabel("Number of Hours")
plt.tight_layout()
plt.show()

### 11.7 Correlation heatmap

Correlation shows how two numeric variables move together. A high positive correlation means they tend to increase together; correlation does not by itself prove causation.

In [ ]:
corr_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

plt.figure(figsize=(10, 7))
sns.heatmap(hourly[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Electrical Measurements")
plt.tight_layout()
plt.show()

### 11.8 Sub-metering comparison

In [ ]:
sub_cols = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
sub_means = hourly[sub_cols].mean()

display(sub_means.to_frame("Average_Hourly_Value"))

plt.figure(figsize=(9, 5))
sub_means.plot(kind="bar")
plt.title("Average Hourly Sub-Metering Values")
plt.xlabel("Sub-Meter")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Feature Engineering

Feature engineering means creating useful input columns from existing information.

For time-series prediction, we use:
- Hour
- Day of week
- Month
- Weekend indicator
- Previous hour demand (`Lag_1`)
- Demand 24 hours earlier (`Lag_24`)
- Demand 168 hours earlier (`Lag_168`, approximately one week)
- Previous 24-hour rolling mean (`Rolling_24`) shifted by one hour

The shift is important: it ensures the model does not accidentally use the current target value to predict itself.

In [ ]:
model_df = hourly[["Global_active_power"]].copy()

model_df["Hour"] = model_df.index.hour
model_df["Day_of_Week"] = model_df.index.dayofweek
model_df["Month"] = model_df.index.month
model_df["Year"] = model_df.index.year
model_df["Is_Weekend"] = (model_df["Day_of_Week"] >= 5).astype(int)

model_df["Lag_1"] = model_df["Global_active_power"].shift(1)
model_df["Lag_24"] = model_df["Global_active_power"].shift(24)
model_df["Lag_168"] = model_df["Global_active_power"].shift(168)

model_df["Rolling_24"] = (
    model_df["Global_active_power"]
    .shift(1)
    .rolling(24)
    .mean()
)

model_df = model_df.dropna().copy()

features = [
    "Hour",
    "Day_of_Week",
    "Month",
    "Year",
    "Is_Weekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "Rolling_24"
]
target_col = "Global_active_power"

X = model_df[features]
y = model_df[target_col]

print("Feature columns:", features)
print("Modeling rows:", len(model_df))
display(model_df.head())

## 13. Chronological Train/Test Split

Because this is time-related data, we do not randomly shuffle it.

The first 80% of observations are used for training and the last 20% are used for testing. This better represents the real situation: learn from the past and predict a later period.

**Data leakage** means allowing information from the future or test set to influence training. That can make a model look better than it really is.

In [ ]:
split_index = int(len(model_df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training period:", X_train.index.min(), "to", X_train.index.max())
print("Testing period:", X_test.index.min(), "to", X_test.index.max())

## 14. Machine Learning

### Regression
Regression is used when the value we want to predict is a number.

### Linear Regression
Linear Regression tries to learn a simple mathematical relationship between the input features and the target.

### Random Forest Regressor
Random Forest uses many decision trees and combines their predictions. It can learn more complex relationships than a simple linear model.

We compare both models using actual evaluation results.

In [ ]:
linear_model = LinearRegression()
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

linear_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

print("Both models trained successfully.")

## 15. Model Evaluation

- **MAE:** Average size of prediction error. Lower is better.
- **RMSE:** Similar to MAE, but gives larger errors more penalty. Lower is better.
- **R²:** Indicates how much variation in the target is explained by the model. Higher is generally better.

In [ ]:
def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

results = pd.DataFrame([
    evaluate_model("Linear Regression", y_test, linear_pred),
    evaluate_model("Random Forest Regressor", y_test, rf_pred)
])

display(results.sort_values(["RMSE", "MAE"]).reset_index(drop=True))

## 16. Actual vs Predicted Consumption

A prediction is useful when predicted values follow the pattern of actual values.

The plot below uses the first part of the test period for readability. The numerical evaluation still uses the complete test set.

In [ ]:
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Linear_Regression": linear_pred,
    "Random_Forest": rf_pred
}, index=y_test.index)

display(comparison.head(20))

plot_n = min(500, len(comparison))

plt.figure(figsize=(14, 5))
plt.plot(comparison.index[:plot_n], comparison["Actual"].iloc[:plot_n], label="Actual")
plt.plot(comparison.index[:plot_n], comparison["Linear_Regression"].iloc[:plot_n], label="Linear Regression")
plt.plot(comparison.index[:plot_n], comparison["Random_Forest"].iloc[:plot_n], label="Random Forest")
plt.title("Actual vs Predicted Electricity Demand")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")
plt.legend()
plt.tight_layout()
plt.show()

## 17. Prediction Error Sample

The error is calculated as:

**Prediction Error = Actual − Predicted**

This table helps inspect individual predictions.

In [ ]:
best_model_name = results.sort_values("RMSE").iloc[0]["Model"]

if best_model_name == "Random Forest Regressor":
    best_predictions = rf_pred
else:
    best_predictions = linear_pred

prediction_sample = pd.DataFrame({
    "Actual_kW": y_test.iloc[:20].values,
    "Predicted_kW": best_predictions[:20]
}, index=y_test.index[:20])

prediction_sample["Error_kW"] = (
    prediction_sample["Actual_kW"] - prediction_sample["Predicted_kW"]
)

display(prediction_sample)

## 18. Automatically Generated Results Summary

This section prints the actual values needed for the report. Copying values from this output into the report ensures that no model performance is invented.

In [ ]:
best_row = results.sort_values("RMSE").iloc[0]

print("===== DATA SUMMARY =====")
print("Original rows:", raw.shape[0])
print("Original columns:", raw.shape[1])
print("Hourly rows used for analysis:", len(hourly))

print("\n===== TARGET SUMMARY =====")
print(f"Mean hourly active power: {target.mean():.4f} kW")
print(f"Maximum hourly active power: {target.max():.4f} kW")
print(f"Minimum hourly active power: {target.min():.4f} kW")

peak_hour = int(avg_by_hour.idxmax())
low_hour = int(avg_by_hour.idxmin())
print(f"Highest average-demand hour: {peak_hour:02d}:00")
print(f"Lowest average-demand hour: {low_hour:02d}:00")

print("\n===== MODEL RESULTS =====")
display(results)

print("\n===== MODEL WITH LOWEST RMSE =====")
print(best_row["Model"])

## 19. Key Insights

Run the notebook first. Then use the generated values and graphs to write the final findings.

Do not write statements such as "Random Forest is better" unless the actual model comparison supports it.

Recommended insight categories:
- Highest and lowest average-demand hours
- Monthly pattern
- Weekday/weekend difference
- Important relationships among electrical measurements
- Model performance according to MAE, RMSE, and R²

## 20. Practical Recommendations

Recommendations should be connected to the observed data.

Examples of reasonable recommendations:
- Where a clear peak period is observed, consider shifting non-essential high-power activities to lower-demand periods when practical.
- Use historical demand patterns to support household energy planning.
- Monitor recurring high-demand periods rather than relying only on daily totals.
- For future systems, combine consumption history with additional information such as weather and tariff data.

These are general recommendations. The final report should be adjusted to the actual findings from this notebook.

## 21. Limitations

1. The dataset represents one household, so results should not be treated as representative of every household.
2. The dataset does not provide all possible demand drivers such as weather, electricity tariff, occupancy, or appliance schedules.
3. The prediction task uses historical demand features and calendar information; it is not a full grid-level forecasting system.
4. Model performance depends on the chosen features, time aggregation, and evaluation period.

## 22. Future Enhancements

- Add weather information.
- Add electricity tariff information.
- Compare additional time-series forecasting methods.
- Build a simple dashboard.
- Deploy the model as a small web application.
- Test the approach on multiple households or larger electricity datasets.

## 23. Conclusion

This project demonstrates an end-to-end data analytics and machine-learning workflow for household electricity demand. It covers data collection, cleaning, preprocessing, exploratory analysis, visualization, feature engineering, chronological model training, evaluation, and prediction.

The final numerical conclusions should be taken directly from the executed notebook outputs.

## 24. References

1. Hebrail, G., & Berard, A. (2006). Individual Household Electric Power Consumption. UCI Machine Learning Repository. Dataset 235. https://doi.org/10.24432/C58K54
2. UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/235/individualhouseholdelectricpowerconsumption
3. Scikit-learn documentation: https://scikit-learn.org/